In [1]:
import polars
import polars as pl
import numpy as np
DATA_BASE_PATH = "../data/"

In [2]:
data = polars.read_csv(DATA_BASE_PATH + "node_information.csv")

In [3]:
# Read txt file containing the edges 
edges_df = polars.read_csv(DATA_BASE_PATH + "train.txt", separator=" ", has_header=False, new_columns=["a", "b", "label"])
edges_df.shape

(10496, 3)

### Feature source

In [4]:
# Create node_feature_mapping from node_id to feature vector (BoW)
node_table = polars.read_csv(DATA_BASE_PATH + "node_information.csv")

node_feature_mapping = {}
for row in node_table.iter_rows():
    node_id = int(row[0])
    features = np.array(row[1:], dtype=np.float32)
    node_feature_mapping[node_id] = features

print("Node features set to BoW:", len(node_feature_mapping), "nodes")

Node features set to BoW: 3596 nodes


Standard interface to create dataset

In [5]:
def build_dataset(edges, labels, feature_builder):
    X = []
    Y = []
    unseen_nodes_count = 0
    for i in range(edges.shape[0]):
        a, b = edges[i]
        edge_features = feature_builder(a, b)
        if edge_features is not None:
            X.append(edge_features)
            Y.append(labels[i])
        else:
            unseen_nodes_count += 1
    print(f"Rows with >=1 unseen node: {unseen_nodes_count} / {edges.shape[0]}")
    return np.array(X), np.array(Y)

### Feature engineering

In [6]:
import math
import networkx as nx

# Graphe des liens positifs pour extraire des features topologiques
G_raw_topo = nx.Graph()
for row in edges_df.filter(pl.col("label") == 1).iter_rows(named=True):
    G_raw_topo.add_edge(int(row["a"]), int(row["b"]))

def topology_features(node_a, node_b):
    a = int(node_a)
    b = int(node_b)
    if a not in G_raw_topo or b not in G_raw_topo:
        return np.array([0.0, 0.0, 0.0], dtype=np.float32)

    common = list(nx.common_neighbors(G_raw_topo, a, b))
    cn = float(len(common))
    aa = 0.0
    ra = 0.0

    for z in common:
        deg_z = G_raw_topo.degree(z)
        if deg_z > 1:
            aa += 1.0 / math.log(deg_z)
        if deg_z > 0:
            ra += 1.0 / deg_z

    return np.array([cn, aa, ra], dtype=np.float32)

def raw_feature_builder(node_a, node_b):
    features_a = node_feature_mapping.get(node_a)
    features_b = node_feature_mapping.get(node_b)
    if features_a is None or features_b is None:
        return None
    topo = topology_features(node_a, node_b)
    return np.concatenate([features_a, features_b])

In [7]:
data = edges_df.to_numpy()
edges = data[:, :2]  # columns: a, b
labels = data[:, 2]  # column: label

X, Y = build_dataset(edges, labels, raw_feature_builder)
# X, Y = build_dataset(edges, labels, pca_feature_builder)

# X, Y = raw_diff_dataset()
print(X.shape, Y.shape)

# Split test and train (stratifie pour conserver la proportion des classes)
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.3, random_state=42, stratify=Y
)

Rows with >=1 unseen node: 7 / 10496
(10489, 1864) (10489,)


In [8]:
from sklearn.metrics import classification_report, roc_auc_score

def evaluate_model(model, x, y):
    y_predicted = model.predict(x)
    report = classification_report(y, y_predicted)

    # AUC sur probabilites si disponibles; fallback sur labels sinon
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(x)[:, 1]
    else:
        y_score = y_predicted

    auc_roc = roc_auc_score(y, y_score)
    print(report)
    print(f"AUC-ROC: {auc_roc:.4f}")

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import networkx as nx

edges_np = edges_df.to_numpy()
edges_train_np, edges_test_np = train_test_split(
    edges_np,
    test_size=0.1,
    random_state=42,
    shuffle=True,
    stratify=edges_np[:, 2],
)

edges_train = pl.DataFrame(edges_train_np, schema=["a", "b", "label"]).with_columns(
    pl.col("a").cast(pl.Int64),
    pl.col("b").cast(pl.Int64),
    pl.col("label").cast(pl.Int64),
)
edges_test = pl.DataFrame(edges_test_np, schema=["a", "b", "label"]).with_columns(
    pl.col("a").cast(pl.Int64),
    pl.col("b").cast(pl.Int64),
    pl.col("label").cast(pl.Int64),
)

print("train/test sizes:", edges_train.shape, edges_test.shape)

train_pos = edges_train.filter(pl.col("label") == 1)
unique_nodes = set(train_pos["a"].to_list()) | set(train_pos["b"].to_list())
node_mapping = {node_id: i for i, node_id in enumerate(sorted(unique_nodes))}

G = nx.Graph()
for row in train_pos.iter_rows(named=True):
    u = node_mapping[row["a"]]
    v = node_mapping[row["b"]]
    G.add_edge(u, v)

print(f"Graph nodes (train positives): {G.number_of_nodes()}")
print(f"Graph edges (train positives): {G.number_of_edges()}")


def get_reduced_node_feature(node_id):
    features = reduced_node_feature_mapping.get(node_id)
    return features






train/test sizes: (9446, 3) (1050, 3)
Graph nodes (train positives): 3428
Graph edges (train positives): 4723


In [ ]:
# Baseline topologique sans fuite: CN, AA, RA + XGBoost
import math
from xgboost import XGBClassifier

def pair_topology_features(node_u, node_v):
    u = node_mapping.get(int(node_u))
    v = node_mapping.get(int(node_v))
    if u is None or v is None:
        return [0.0, 0.0, 0.0]

    common = list(nx.common_neighbors(G, u, v))
    cn = float(len(common))

    aa = 0.0
    ra = 0.0
    for z in common:
        deg_z = G.degree(z)
        if deg_z > 1:
            aa += 1.0 / math.log(deg_z)
        if deg_z > 0:
            ra += 1.0 / deg_z

    return [cn, aa, ra]


def make_pair_dataset(edges_df, pair_builder):
    X = []
    y = []
    for row in edges_df.iter_rows(named=True):
        a = int(row["a"])

        b = int(row["b"])

        label = int(row["label"])

        X.append(pair_builder(a, b))
        y.append(label)
    return np.array(X, dtype=np.float32), np.array(y)





: 

In [ ]:
# MLP avec selection standardisee des features via une liste (version riche)
import math
import copy
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# -------------------------------------------------------------
# 1) Registry de features + presets
# -------------------------------------------------------------
NODE_DIM = len(next(iter(node_feature_mapping.values())))

# Precompute once for centrality-based features
pagerank_map = nx.pagerank(G_raw_topo, alpha=0.85)
katz_map = nx.katz_centrality(G_raw_topo, alpha=0.005, beta=1.0, max_iter=1000, tol=1e-6)

# Precompute clustering coefficients (dense graph)
try:
    clustering_coefs = nx.clustering(G_raw_topo)
except:
    clustering_coefs = {}
    
# Precompute triangle count for each node
triangles = {node: 0 for node in G_raw_topo.nodes()}
for node in G_raw_topo.nodes():
    neighbors = set(G_raw_topo.neighbors(node))
    for n1, n2 in [(n1, n2) for i, n1 in enumerate(neighbors) for n2 in list(neighbors)[i+1:]]:
        if G_raw_topo.has_edge(n1, n2):
            triangles[node] += 1


def get_node_feat(node_id):
    return node_feature_mapping.get(int(node_id))


def common_neighbors(a, b, graph):
    if a not in graph or b not in graph:
        return []
    return list(nx.common_neighbors(graph, a, b))


# Node-wise feature functions

def f_cosine_similarity(a, b, fa, fb, graph):
    denom = (np.linalg.norm(fa) * np.linalg.norm(fb)) + 1e-8
    return float(np.dot(fa, fb) / denom)


def f_dot_product(a, b, fa, fb, graph):
    return float(np.dot(fa, fb))


def f_l2_distance(a, b, fa, fb, graph):
    return float(np.linalg.norm(fa - fb))


# Topological feature functions (raw)

def f_common_neighbors(a, b, fa, fb, graph):
    return float(len(common_neighbors(a, b, graph)))


def f_adamic_adar(a, b, fa, fb, graph):
    score = 0.0
    for z in common_neighbors(a, b, graph):
        deg_z = graph.degree(z)
        if deg_z > 1:
            score += 1.0 / math.log(deg_z)
    return float(score)


def f_resource_allocation(a, b, fa, fb, graph):
    score = 0.0
    for z in common_neighbors(a, b, graph):
        deg_z = graph.degree(z)
        if deg_z > 0:
            score += 1.0 / deg_z
    return float(score)


def f_jaccard(a, b, fa, fb, graph):
    if a not in graph or b not in graph:
        return 0.0
    na = set(graph.neighbors(a))
    nb = set(graph.neighbors(b))
    union = len(na | nb)
    return float(len(na & nb) / union) if union > 0 else 0.0


def f_preferential_attachment(a, b, fa, fb, graph):
    if a not in graph or b not in graph:
        return 0.0
    return float(graph.degree(a) * graph.degree(b))


def f_degree_a(a, b, fa, fb, graph):
    if a not in graph:
        return 0.0
    return float(graph.degree(a))


def f_degree_b(a, b, fa, fb, graph):
    if b not in graph:
        return 0.0
    return float(graph.degree(b))


def f_degree_diff(a, b, fa, fb, graph):
    if a not in graph or b not in graph:
        return 0.0
    return float(abs(graph.degree(a) - graph.degree(b)))


# Topological/structural feature functions (log-compressed)

def f_adamic_adar_log1p(a, b, fa, fb, graph):
    return float(np.log1p(f_adamic_adar(a, b, fa, fb, graph)))


def f_resource_allocation_log1p(a, b, fa, fb, graph):
    return float(np.log1p(f_resource_allocation(a, b, fa, fb, graph)))


def f_preferential_attachment_log1p(a, b, fa, fb, graph):
    return float(np.log1p(f_preferential_attachment(a, b, fa, fb, graph)))


def f_degree_a_log1p(a, b, fa, fb, graph):
    return float(np.log1p(f_degree_a(a, b, fa, fb, graph)))


def f_degree_b_log1p(a, b, fa, fb, graph):
    return float(np.log1p(f_degree_b(a, b, fa, fb, graph)))


def f_degree_diff_log1p(a, b, fa, fb, graph):
    return float(np.log1p(f_degree_diff(a, b, fa, fb, graph)))


# Centrality feature functions

def f_pagerank_a_log1p(a, b, fa, fb, graph):
    return float(np.log1p(pagerank_map.get(int(a), 0.0)))


def f_pagerank_b_log1p(a, b, fa, fb, graph):
    return float(np.log1p(pagerank_map.get(int(b), 0.0)))


def f_pagerank_diff_log1p(a, b, fa, fb, graph):
    pa = float(pagerank_map.get(int(a), 0.0))
    pb = float(pagerank_map.get(int(b), 0.0))
    return float(np.log1p(abs(pa - pb)))


def f_katz_a_log1p(a, b, fa, fb, graph):
    return float(np.log1p(katz_map.get(int(a), 0.0)))


def f_katz_b_log1p(a, b, fa, fb, graph):
    return float(np.log1p(katz_map.get(int(b), 0.0)))


def f_katz_diff_log1p(a, b, fa, fb, graph):
    ka = float(katz_map.get(int(a), 0.0))
    kb = float(katz_map.get(int(b), 0.0))
    return float(np.log1p(abs(ka - kb)))


# Structural features avec clustering et triangles

def f_clustering_coef_a(a, b, fa, fb, graph):
    return float(clustering_coefs.get(int(a), 0.0))


def f_clustering_coef_b(a, b, fa, fb, graph):
    return float(clustering_coefs.get(int(b), 0.0))


def f_triangle_count_a(a, b, fa, fb, graph):
    return float(triangles.get(int(a), 0))


def f_triangle_count_b(a, b, fa, fb, graph):
    return float(triangles.get(int(b), 0))


def f_salton_index(a, b, fa, fb, graph):
    deg_a = graph.degree(int(a)) if int(a) in graph else 0
    deg_b = graph.degree(int(b)) if int(b) in graph else 0
    if deg_a == 0 or deg_b == 0:
        return 0.0
    cn = float(len(common_neighbors(int(a), int(b), graph)))
    return cn / (np.sqrt(deg_a * deg_b) + 1e-8)


def f_sorensen_index(a, b, fa, fb, graph):
    deg_a = graph.degree(int(a)) if int(a) in graph else 0
    deg_b = graph.degree(int(b)) if int(b) in graph else 0
    if deg_a + deg_b == 0:
        return 0.0
    cn = float(len(common_neighbors(int(a), int(b), graph)))
    return (2.0 * cn) / (deg_a + deg_b)


def f_salton_index_log1p(a, b, fa, fb, graph):
    return float(np.log1p(f_salton_index(a, b, fa, fb, graph)))


def f_sorensen_index_log1p(a, b, fa, fb, graph):
    return float(np.log1p(f_sorensen_index(a, b, fa, fb, graph)))


FEATURE_REGISTRY = {
    # Node-wise
    "cosine_similarity": f_cosine_similarity,
    "dot_product": f_dot_product,
    "l2_distance": f_l2_distance,
    # Topological raw
    "common_neighbors": f_common_neighbors,
    "adamic_adar": f_adamic_adar,
    "resource_allocation": f_resource_allocation,
    "jaccard": f_jaccard,
    "preferential_attachment": f_preferential_attachment,
    "degree_a": f_degree_a,
    "degree_b": f_degree_b,
    "degree_diff": f_degree_diff,
    # Topological transformed
    "adamic_adar_log1p": f_adamic_adar_log1p,
    "resource_allocation_log1p": f_resource_allocation_log1p,
    "preferential_attachment_log1p": f_preferential_attachment_log1p,
    "degree_a_log1p": f_degree_a_log1p,
    "degree_b_log1p": f_degree_b_log1p,
    "degree_diff_log1p": f_degree_diff_log1p,
    # Centrality
    "pagerank_a_log1p": f_pagerank_a_log1p,
    "pagerank_b_log1p": f_pagerank_b_log1p,
    "pagerank_diff_log1p": f_pagerank_diff_log1p,
    "katz_a_log1p": f_katz_a_log1p,
    "katz_b_log1p": f_katz_b_log1p,
    "katz_diff_log1p": f_katz_diff_log1p,
    # Structural (clustering, triangles, indices)
    "clustering_coef_a": f_clustering_coef_a,
    "clustering_coef_b": f_clustering_coef_b,
    "triangle_count_a": f_triangle_count_a,
    "triangle_count_b": f_triangle_count_b,
    "salton_index": f_salton_index,
    "sorensen_index": f_sorensen_index,
    "salton_index_log1p": f_salton_index_log1p,
    "sorensen_index_log1p": f_sorensen_index_log1p,
}

FEATURE_PRESETS = {
    "0.83": [
        "resource_allocation_log1p",
        "jaccard",
        "preferential_attachment_log1p",
        "degree_a_log1p",
        "degree_b_log1p",
        "degree_diff_log1p",
        "cosine_similarity",
        # "pagerank_a_log1p",
        # "pagerank_b_log1p",
        # "pagerank_diff_log1p",
        "katz_a_log1p",
        "katz_b_log1p",
        "katz_diff_log1p",
    ],
    "baseline_small": [
        "resource_allocation_log1p",
        "jaccard",
        "preferential_attachment_log1p",
        "degree_a_log1p",
        "degree_b_log1p",
        "degree_diff_log1p",
        "cosine_similarity",
        # "pagerank_a_log1p",
        # "pagerank_b_log1p",
        # "pagerank_diff_log1p",
        "katz_a_log1p",
        "katz_b_log1p",
        "katz_diff_log1p",
    ],
    "legacy_best_like": [
        "resource_allocation_log1p",
        "jaccard",
        "preferential_attachment_log1p",
        "degree_a_log1p",
        "degree_b_log1p",
        "degree_diff_log1p",
        "cosine_similarity",
        "pagerank_a_log1p",
        "pagerank_b_log1p",
        "pagerank_diff_log1p",
        "katz_a_log1p",
        "katz_b_log1p",
        "katz_diff_log1p",
        "clustering_coef_a",
        "clustering_coef_b",
        "triangle_count_a",
        "triangle_count_b",
        "salton_index",
        "sorensen_index",
        "salton_index_log1p",
        "sorensen_index_log1p",
    ],
    "all": list(FEATURE_REGISTRY.keys()),
}

# Choix simple: preset puis liste finale derivee
ACTIVE_PRESET = "baseline_small"
SELECTED_FEATURES = FEATURE_PRESETS[ACTIVE_PRESET]

unknown = [name for name in SELECTED_FEATURES if name not in FEATURE_REGISTRY]
if unknown:
    raise ValueError(f"Unknown features in SELECTED_FEATURES: {unknown}")

print("Active preset:", ACTIVE_PRESET)
print("Selected features:", SELECTED_FEATURES)


# -------------------------------------------------------------
# 2) Build dataset depuis la liste de features selectionnees
# -------------------------------------------------------------
DROP_UNSEEN_NODE_PAIRS = True


def build_feature_vector(a, b, selected_features, graph):
    fa = get_node_feat(a)
    fb = get_node_feat(b)
    if fa is None or fb is None:
        return None
    values = [FEATURE_REGISTRY[name](a, b, fa, fb, graph) for name in selected_features]
    return np.array(values, dtype=np.float32)


def build_selected_feature_dataset(edges_array, selected_features, graph):
    X_out = []
    y_out = []
    skipped = 0
    for i in range(edges_array.shape[0]):
        a, b, y = edges_array[i]
        vec = build_feature_vector(int(a), int(b), selected_features, graph)
        if vec is None:
            if DROP_UNSEEN_NODE_PAIRS:
                skipped += 1
                continue
            vec = np.zeros(len(selected_features), dtype=np.float32)
        X_out.append(vec)
        y_out.append(int(y))
    print(f"Skipped rows with unseen node features: {skipped}")
    return np.array(X_out, dtype=np.float32), np.array(y_out, dtype=np.int64)


edges_all = edges_df.to_numpy()
X_selected, y_selected = build_selected_feature_dataset(edges_all, SELECTED_FEATURES, G_raw_topo)

print("Dataset shape:", X_selected.shape, y_selected.shape)
print("Feature count used by MLP:", X_selected.shape[1])


# -------------------------------------------------------------
# 3) Train / validation / test split (60 / 20 / 20)
# -------------------------------------------------------------
# First split: 80% (train+val) / 20% (test)
X_temp, X_test, y_temp, y_test = train_test_split(
    X_selected, y_selected, test_size=0.2, random_state=42, stratify=y_selected
)

# Second split: 75% (train) / 25% (val) of temp => 60% train, 20% val overall
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}, Val shape: {X_val.shape}")

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train).astype(np.float32)
X_val_s = scaler.transform(X_val).astype(np.float32)
X_test_s = scaler.transform(X_test).astype(np.float32)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

train_ds = TensorDataset(
    torch.tensor(X_train_s, dtype=torch.float32),
    torch.tensor(y_train.astype(np.float32), dtype=torch.float32),
)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
X_val_t = torch.tensor(X_val_s, dtype=torch.float32).to(device)
X_test_t = torch.tensor(X_test_s, dtype=torch.float32).to(device)


class MLPSelectedFeatures(nn.Module):
    def __init__(self, in_dim, proj_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, proj_dim),
            nn.BatchNorm1d(proj_dim),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(proj_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x)


model = MLPSelectedFeatures(in_dim=X_train_s.shape[1], proj_dim=128).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=0)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=4,
    min_lr=1e-5,
)

max_epochs = 200
patience = 50
losses = []
val_aucs = []
best_auc = -1.0
best_epoch = 0
best_state = None
wait = 0

for epoch in range(1, max_epochs + 1):
    model.train()
    run_loss = 0.0
    n = 0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        logits = model(xb).squeeze(1)
        loss = criterion(logits, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        bs = xb.size(0)
        run_loss += loss.item() * bs
        n += bs

    train_loss = run_loss / max(n, 1)

    model.eval()
    with torch.no_grad():
        val_logits = model(X_val_t).squeeze(1)
        val_proba = torch.sigmoid(val_logits).detach().cpu().numpy()
    val_auc = roc_auc_score(y_val, val_proba)
    scheduler.step(val_auc)
    current_lr = optimizer.param_groups[0]["lr"]

    losses.append(train_loss)
    val_aucs.append(val_auc)

    if val_auc > best_auc + 1e-5:
        best_auc = val_auc
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        wait = 0
    else:
        wait += 1

    clear_output(wait=True)
    fig, ax1 = plt.subplots(figsize=(8, 4))
    ax1.plot(losses, color="tab:blue")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss", color="tab:blue")
    ax1.tick_params(axis="y", labelcolor="tab:blue")

    ax2 = ax1.twinx()
    ax2.plot(val_aucs, color="tab:orange")
    ax2.set_ylabel("AUC", color="tab:orange")
    ax2.tick_params(axis="y", labelcolor="tab:orange")

    if best_epoch > 0:
        ax2.scatter([best_epoch], [best_auc], color="tab:red", s=36)

    plt.title(f"MLP selected features ({ACTIVE_PRESET}) - epoch {epoch}/{max_epochs}")
    plt.tight_layout()
    plt.show()

    print(
        f"Device: {device} | Epoch {epoch:03d} | loss={train_loss:.6f} | lr={current_lr:.2e} | "
        f"val_auc={val_auc:.6f} | best_auc={best_auc:.6f} (epoch {best_epoch})"
    )

    if wait >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

if best_state is not None:
    model.load_state_dict(best_state)

# Evaluation on validation set
model.eval()
with torch.no_grad():
    val_logits = model(X_val_t).squeeze(1)
    y_val_proba = torch.sigmoid(val_logits).detach().cpu().numpy()
y_val_pred = (y_val_proba >= 0.5).astype(int)

print("\n" + "="*80)
print("VALIDATION SET RESULTS")
print("="*80)
print(f"Best epoch: {best_epoch}")
print(classification_report(y_val, y_val_pred))
print(f"Validation AUC-ROC: {roc_auc_score(y_val, y_val_proba):.4f}")

# Evaluation on test set
with torch.no_grad():
    test_logits = model(X_test_t).squeeze(1)
    y_test_proba = torch.sigmoid(test_logits).detach().cpu().numpy()
y_test_pred = (y_test_proba >= 0.5).astype(int)

print("\n" + "="*80)
print("TEST SET RESULTS")
print("="*80)
print(classification_report(y_test, y_test_pred))
print(f"Test AUC-ROC: {roc_auc_score(y_test, y_test_proba):.4f}")

In [ ]:
# Submission: Load test set, build features, predict, save
print("=" * 80)
print("SUBMISSION GENERATION")
print("=" * 80)

test_df = pl.read_csv(
    DATA_BASE_PATH + "test.txt", 
    separator=" ", 
    has_header=False, 
    new_columns=["a", "b"]
)

test_np = test_df.to_numpy()
X_test_selected = []

for i in range(test_np.shape[0]):
    a, b = test_np[i]
    vec = build_feature_vector(int(a), int(b), SELECTED_FEATURES, G_raw_topo)
    if vec is None:
        vec = np.zeros(len(SELECTED_FEATURES), dtype=np.float32)
    X_test_selected.append(vec)

X_test_selected = np.array(X_test_selected, dtype=np.float32)
print(f"Test set shape: {X_test_selected.shape}")

# Scale with training scaler
X_test_scaled = scaler.transform(X_test_selected).astype(np.float32)

# Predict
model.eval()
with torch.no_grad():
    x_test_t = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
    logits_test = model(x_test_t).squeeze(1)
    proba_test = torch.sigmoid(logits_test).cpu().numpy()

pred_test = (proba_test >= 0.5).astype(int)

# Save submissions
submission_proba = pl.DataFrame({
    "ID": np.arange(len(proba_test)), 
    "Predicted": proba_test
})
submission_label = pl.DataFrame({
    "ID": np.arange(len(pred_test)), 
    "Predicted": pred_test
})

proba_file = DATA_BASE_PATH + "submission_mlp_final_proba.csv"
label_file = DATA_BASE_PATH + "submission_mlp_final_label.csv"

submission_proba.write_csv(proba_file)
submission_label.write_csv(label_file)

print(f"✓ Saved: {proba_file}")
print(f"✓ Saved: {label_file}")
print(f"Test predictions shape: {pred_test.shape}")
print(f"Positive predictions: {pred_test.sum()} / {len(pred_test)} ({100*pred_test.sum()/len(pred_test):.1f}%)")
print(f"Proba range: [{proba_test.min():.4f}, {proba_test.max():.4f}]")
print(f"Mean probability: {proba_test.mean():.4f}")
submission_proba.head()


SUBMISSION GENERATION
Test set shape: (3498, 13)
✓ Saved: data/submission_mlp_final_proba.csv
✓ Saved: data/submission_mlp_final_label.csv
Test predictions shape: (3498,)
Positive predictions: 1624 / 3498 (46.4%)
Proba range: [0.0000, 1.0000]
Mean probability: 0.4925


ID,Predicted
i64,f32
0,0.474108
1,0.300426
2,0.423093
3,0.503078
4,0.10095
